# Download các thư viện cần thiết

In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [2]:
def read_parquet_user(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [3]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [4]:
def read_parquet_transaction(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [5]:
def split_and_save_parquet(df, num_files, output_dir, type):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.{type}_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Chuẩn bị dữ liệu

In [6]:
user = read_parquet_user("./preprocessed-dataset")
user.head()

customer_id,gender,province,membership,created_date,last_sync_date,install_app,install_datetime,user_age_days,days_since_install,days_since_last_sync
i32,str,str,str,date,date,str,date,f64,f64,f64
6359043,"""male""","""Kon Tum""","""Standard""",2023-04-09,2025-07-16,"""In-Store""",2023-04-09,905.161726,905.917091,76.414299
6359044,"""male""","""Bình Thuận""","""Gold""",2023-04-09,2025-07-16,"""In-Store""",2023-04-09,905.161344,905.917091,76.414299
6359046,"""female""","""Hồ Chí Minh""","""Standard""",2023-04-09,2025-07-16,"""In-Store""",2023-04-09,905.161024,905.917091,76.414299
6359051,"""female""","""Nghệ An""","""Standard""",2023-04-09,2025-07-16,"""In-Store""",2023-04-09,905.160259,905.917091,76.414299
6359055,"""male""","""Đồng Nai""","""Standard""",2023-04-09,2025-07-16,"""In-Store""",2023-04-09,905.159724,905.917091,76.414299


In [7]:
transaction = read_parquet_transaction("./preprocessed-dataset")
transaction.head()

item_id,price,quantity,customer_id,created_date,channel,payment,location,discount,list_price,category_l2,discount_rate
str,"decimal[38,4]",i32,i32,date,str,str,i32,"decimal[38,4]","decimal[38,4]",str,"decimal[38,4]"
"""2596000000003""",296000.0000,1,7627794,2024-12-12,"""iOS""","""Tiền mặt""",578,149000.0000,445000.0000,"""Moony""",0.3348
"""1512000000004""",274349.0000,1,7367484,2024-12-12,"""SPE""","""Tiền mặt""",945,20651.0000,295000.0000,"""TPCN cho bé""",0.0700
"""5537000000015""",70562.0000,1,4795945,2024-12-12,"""SPE""","""Không xác định""",321,4438.0000,75000.0000,"""Snack ăn dặm""",0.0592
"""4644000000001""",58240.0000,4,7253028,2024-12-12,"""In-Store""","""Tiền mặt""",304,23040.0000,65000.0000,"""Caryn""",0.1040
"""2700000000002""",61750.0000,1,7681758,2024-12-12,"""SPE""","""Tiền mặt""",735,3250.0000,65000.0000,"""Khăn khô""",0.0500


In [8]:
item = read_parquet_item("./preprocessed-dataset")
item.head()

item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,item_type,description_new,gender_target_final,age_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Không xác định""","""Từ 9M"""
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Bé Gái""","""Bộ quần áo""","""Không xác định""","""Bé Gái""","""Từ 36M"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm…","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm …","""Không xác định""","""0-12M"""
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miế…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""[""Từ 4M"", ""3M-6M"", ""12-36M""]"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M …","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""12-36M"""


# Training Stage 1